In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hypothesisTesting.ipynb")

## Lecture Section

In this lecture, we will cover more aspects of hypothesis testing. We will rely on the 'statsmodels' library for most of this lecture-assignment.

We will cover:
* ANOVAs
* Power
* z-tests
* * Contingency Tables

In the previous lecture, we learned how to do a one-way ANOVA with `scipy`.

### ANOVA (again!)

We are going to use the `ToothGrowth` dataset from `statsmodels` - it's an R dataset of supplement dosing and tooth growth.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.libqsturng import psturng
from statsmodels.stats.multicomp import MultiComparison
from statsmodels.imputation.mice import MICEData
import numpy as np


# Let's load the `ToothGrowth` dataset and examine whether supplement type and dose affect tooth length.
df2 = sm.datasets.get_rdataset("ToothGrowth", "datasets").data
df2["dose"] = df2["dose"].astype("category")
df2.head()


### ANOVA

Let's start with ANOVAs. We already covered a one-way test with `scipy`, but we can do the same with `statsmodels`. Let's test whether supplement type affects tooth length. First, we create a linear regression model with `smf.ols()`. Then, we use `statsmodels.api as sm` to call `anova_lm()`. We use `typ=2` to denote that there are no interaction effects.

In [ ]:
model = smf.ols("len ~ C(dose)", data=df2).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

### 2-Way Anova

We do the same for a two-way ANOVA. We add the other predictor variable to our regression model, then run the `.anova_lm()` function again. Notice the interaction term we use: `C(supp):C(dose)` - we use it to show how the effect of the supplement depends on the dose.

The [T.VC], [T.1.0], etc. in the rest of our results represent the other categories within our dataset. They are dummy variables!

In [ ]:
model2 = smf.ols("len ~ C(supp) + C(dose) + C(supp):C(dose)", data=df2).fit()
print(sm.stats.anova_lm(model2, typ=2))
print(model2.summary())

Heading back to the one-way ANOVA... let's get more information. We can run `pairwise_tukeyhsd()` from `statsmodels.stats.multicomp` - it's a type of post-hoc test that makes pairwise comparisons between the means  of each group.

If we compare the mean tooth lengths of each dose group, we see that the differences between all three groups of doses are meaningful!

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
tukey = pairwise_tukeyhsd(endog=df2["len"],
                          groups=df2["dose"],
                          alpha=0.05)
print(tukey)

In [ ]:
sns.boxplot(data=df2, x="dose", y="len", hue="supp")
plt.title("Tooth Growth by Supplement and Dose")
plt.show()

### Power

Let's take a quick detour and talk about statistical power. `statsmodels` has a lot of different functions to help you determine statistical power - we will try a couple.

This first one is a class called TTestIndPower. It holds many methods for determining power of an independant two-sample t-test. `.power()` will find the power.

In [ ]:
from statsmodels.stats.power import TTestIndPower
power_analysis = TTestIndPower()
power = power_analysis.power(effect_size=0.4, nobs1=100, alpha=0.05)
print("Power: ", power)

We can also use `.solve_power()`. This method solves for a missing parameter of your choosing. In the case below, I am solving for sample size to reach a power of 0.8 with a 95% confidence and an effect size of 0.4.

In [ ]:
from statsmodels.stats.power import TTestIndPower
power_analysis = TTestIndPower()
power = power_analysis.solve_power(effect_size=0.4, nobs1=None, alpha=0.05, power = 0.8)
print("Value: ", power)

`statsmodels` has similar functions for normal distributions, chi-square, and ANOVAs.

### z-tests

(We are going to use the `tips` dataset for this portion.)

Finally, let's talk about z-tests in Python. They are similar to t-tests, but they compare to a known population value. We will pretend that average tips nationwide is $3, and we will pretend to know the population variances follow a normal distribution.

To perform a one-sample z-test, we use `statsmodels.stats.weightstats` and import `ztest`. Then, we pass our column of data as the first argument, and our population value as the second.

In [ ]:
tips = sns.load_dataset("tips")
tips.head()

In [ ]:
from statsmodels.stats.weightstats import ztest
overall_mean = 3
z_stat, p_val = ztest(tips["tip"], value=overall_mean)
print("One-sample z-test: z = ", z_stat, " p-value = ", p_val)

For a two-sample z-test, we use the same function but our second argument is just our second column of data.

In [ ]:
male_tips = tips[tips["sex"] == "Male"]["tip"]
female_tips = tips[tips["sex"] == "Female"]["tip"]
z_stat2, p_val2 = ztest(male_tips, female_tips)
print("Two-sample z-test: z = ", z_stat2, " p-value = ", p_val2)

### Contingency Table

We've worked quite a bit without visualizing our categorical variables and their class labels in a contingency table. 'statsmodels' makes this easy for us.

In [ ]:
import pandas as pd
import statsmodels.api as sm

df = sm.datasets.get_rdataset("Arthritis", "vcd").data
df.fillna({"Improved":"None"}, inplace=True)
df.head()

This data shows people with arthritis and whether they had improvement in symptoms after a treatment (or placebo). We want a contingency table of the treatment and the outcome (improvement level). We can use 'sm.stats.Table()' to create a nice table for us.

In [ ]:
tab = pd.crosstab(df['Treatment'], df['Improved'])
tab = tab.loc[:, ["None", "Some", "Marked"]]

table = sm.stats.Table(tab)
table.table_orig

We can get the expected cell counts (assuming independence) and compare to our contingency table

In [ ]:
table.fittedvalues  # Expected values under independence

We can see how much each cell deviates from the expected value, too

In [ ]:
table.resid_pearson

In [ ]:
result = table.test_ordinal_association()
print(result)

If your variables are nominal and not ordinal, you can use the nominal version of the same function. We have an ordinal variable in this case, so our p-value will not be as small, but is still statistically significant under most assumptions of alpha!

In [ ]:
result = table.test_nominal_association()
print(result)

## Assignment Section

**Question 1.** Run an ANOVA on the 'penguins' dataset from seaborn. You want to compare the average 'body_mass_g' for each penguin 'species'. Use the 'statsmodels' library this time.

In [ ]:
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm


penguins = sns.load_dataset("penguins").dropna()

anova_table = ...


In [ ]:
grader.check("q1")

**Question 2.** Create a 2-way ANOVA similar to the above problem, but add the categorical variable 'sex'.

In [ ]:
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm

penguins = sns.load_dataset("penguins").dropna()
model2 = ...
print(sm.stats.anova_lm(model2, typ=2))
print(model2.summary())

In [ ]:
grader.check("q2")

<!-- BEGIN QUESTION -->

**Question 3.** Create a boxplot of 'body_mass_g' (y), 'species' (x), and colored by 'sex'.

In [ ]:
...

<!-- END QUESTION -->

**Question 4.** Let's assume the 'penguins' dataset is representative sample of a population of penguins who's mean body_mass_g is 4200. Do a z-test and calculate whether the mean of our sample differs from the true population mean of 4200.

In [ ]:
from statsmodels.stats.weightstats import ztest
import seaborn as sns
import numpy as np
...
print("One-sample z-test: z = ", z_stat, " p-value = ", p_val)

In [ ]:
grader.check("q4")

**Question 5.** Using the 'penguins' dataset, create a contingency table between 'species' and 'sex'. Also give the fitted values in a table, and run a nominal association test.

In [ ]:
import seaborn as sns
import pandas as pd
...

table = ...
fitted = ...
result = ...



In [ ]:
grader.check("q5")

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()